# Corpus Extraction Batch (Mistral)

Enrichit le corpus `single_task_benchmark_paper.parquet` (1357 papiers single-task) via deux extractions LLM :

1. **Resource links** — arbitrage `link_arbiter` sur candidats PDF/GROBID (dataset/code URLs).
2. **Benchmark languages** — codes ISO 639-3 du texte annoté depuis `title` + `abstract`.

Les résultats intermédiaires sont écrits dans `data/corpus/resource_links.parquet` et `data/corpus/language_extractions.parquet`, puis fusionnés dans le corpus via `build_single_task_benchmark_paper.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import subprocess
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve().parent))

In [2]:
from src.taxonomy.providers import mistral_batch_results, mistral_batch_submit
from src.taxonomy.schemas import TASKS

c:\Users\samma\Documents\Programmation\UdeM\nlp-benchmark-taxonomy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Shared config

In [3]:
MODEL = "mistral-large-latest"

DATA_DIR = Path("../data")
CORPUS_PATH = DATA_DIR / "corpus" / "single_task_benchmark_paper.parquet"
RESOURCE_LINKS_DIR = DATA_DIR / "corpus" / "resource_links"
LINK_BATCH_PATH = RESOURCE_LINKS_DIR / "link_arbiter_batch.parquet"
LINK_OUT_PATH = DATA_DIR / "corpus" / "resource_links.parquet"
LANGUAGE_OUT_PATH = DATA_DIR / "corpus" / "language_extractions.parquet"

LINK_TASK = "link_arbiter"
LANGUAGE_TASK = "language_assignment"

link_cfg = TASKS[LINK_TASK]
language_cfg = TASKS[LANGUAGE_TASK]
print("links:", link_cfg.tool_name, link_cfg.max_tokens)
print("language:", language_cfg.tool_name, language_cfg.max_tokens)

links: arbitrate_resource_links 1024
language: extract_benchmark_languages 256


## 2. Load corpus input

Source : `single_task_benchmark_paper.parquet` (produit par `src/corpus/build_single_task_benchmark_paper.py`).
Requis pour les liens : `anthology_id`, `pdf_url`, `title`, `abstract`. Pour les langues : `bibkey`, `title`, `abstract`.

In [4]:
corpus = pd.read_parquet(CORPUS_PATH)
required = {"bibkey", "anthology_id", "title", "abstract", "pdf_url"}
missing = required - set(corpus.columns)
if missing:
    raise ValueError(f"Colonnes manquantes dans {CORPUS_PATH}: {sorted(missing)}")


print(f"{len(corpus):,} papers")
corpus[["bibkey", "anthology_id", "title"]].head(3)

1,357 papers


,bibkey,anthology_id,title
0,-etal-2020-multi,2020.lrec-1.621,Multi-domain Tweet Corpora for Sentiment Analy...
1,-etal-2022-hindimd,2022.lrec-1.764,HindiMD: A Multi-domain Corpora for Low-resour...
2,abbes-etal-2020-daict,2020.lrec-1.768,DAICT: A Dialectal Arabic Irony Corpus Extract...


## 3. Resource links — prepare candidates

Acquisition PDF + GROBID TEI, extraction des candidats, parquet batch. Input = corpus ; output = `link_arbiter_batch.parquet`.

GROBID doit tourner sur `localhost:8070`. Les caches PDF/TEI sont réutilisables (`--skip-download --skip-grobid` si déjà rempli).

In [5]:
cmd = [
    "uv",
    "run",
    "python",
    "-m",
    "src.extraction.extract_resource_links_pipeline",
    "--input",
    str(CORPUS_PATH.resolve()),
    "--batch",
    str(LINK_BATCH_PATH.resolve()),
    "--skip-download",
    "--skip-grobid",
]
subprocess.run(cmd, check=True, cwd=Path().resolve().parent)

CompletedProcess(args=['uv', 'run', 'python', '-m', 'src.extraction.extract_resource_links_pipeline', '--input', 'C:\\Users\\samma\\Documents\\Programmation\\UdeM\\nlp-benchmark-taxonomy\\data\\corpus\\single_task_benchmark_paper.parquet', '--batch', 'C:\\Users\\samma\\Documents\\Programmation\\UdeM\\nlp-benchmark-taxonomy\\data\\corpus\\resource_links\\link_arbiter_batch.parquet', '--skip-download', '--skip-grobid'], returncode=0)

## 4. Resource links — submit batch

Run once. Keep the printed `link_job_id`.

In [9]:
link_df = pd.read_parquet(LINK_BATCH_PATH)
link_df["bibkey"] = link_df["anthology_id"]
link_df = link_df[link_df["anthology_id"].isin(corpus["anthology_id"])]

print(f"{len(link_df):,} papiers à arbitrer")
link_df[["anthology_id", "n_candidates_raw"]].head(3)

1,357 papiers à arbitrer


,anthology_id,n_candidates_raw
0,2020.lrec-1.621,3
1,2022.lrec-1.764,3
2,2020.lrec-1.768,1


In [10]:
link_items = [
    (str(row.anthology_id), {"paper_text": row.paper_text})
    for row in link_df.itertuples(index=False)
]

link_job_id = mistral_batch_submit(link_items, model=MODEL, task=LINK_TASK)
link_job_id

'b9eab017-d94e-4b2c-aafe-48e50701e583'

## 5. Resource links — fetch results

À lancer quand le batch est terminé. Aplatit `dataset_link` / `code_link` et écrit `resource_links.parquet`.

In [11]:
LINK_JOB_ID = "b9eab017-d94e-4b2c-aafe-48e50701e583"

links = mistral_batch_results(LINK_JOB_ID, link_df, task=LINK_TASK)

# dataset_link / code_link sont des LISTES (miroir HF + GitHub possible).
# -> {kind}_urls = toutes les URLs ; {kind}_url = la 1re (pour l'analyse / le test vide).
for kind in ("dataset", "code"):
    col = f"mistral_{kind}_link"
    items = links[col].apply(lambda v: list(v) if isinstance(v, (list, tuple)) else [])
    links[f"{kind}_urls"] = items.apply(
        lambda xs: [d.get("url") for d in xs if d.get("url")]
    )
    links[f"{kind}_url"] = links[f"{kind}_urls"].apply(lambda us: us[0] if us else None)
    first = items.apply(lambda xs: xs[0] if xs else {})
    for field in ("host_type", "is_official", "confidence"):
        links[f"{kind}_{field}"] = first.apply(lambda d, f=field: d.get(f))
    links = links.drop(columns=col)

links = links.rename(
    columns={
        "mistral_data_availability": "data_availability",
        "mistral_reasoning": "arbiter_reasoning",
    }
)

cols = [
    "anthology_id",
    "n_candidates_raw",
    "dataset_url",
    "dataset_urls",
    "dataset_host_type",
    "dataset_is_official",
    "dataset_confidence",
    "code_url",
    "code_urls",
    "code_host_type",
    "code_is_official",
    "code_confidence",
    "data_availability",
    "arbiter_reasoning",
    "candidates_json",
]
links = links[[c for c in cols if c in links.columns]]
links.to_parquet(LINK_OUT_PATH, index=False)
LINK_OUT_PATH

WindowsPath('../data/corpus/resource_links.parquet')

## 6. Benchmark languages — submit batch

Input = `title` + `abstract` (langue du texte annoté, pas la langue du papier). Run once ; garder `language_job_id`.

In [ ]:
language_items = [
    (
        str(row.bibkey),
        {"title": row.title, "abstract": row.abstract},
    )
    for row in corpus.itertuples(index=False)
]

language_job_id = mistral_batch_submit(language_items, model=MODEL, task=LANGUAGE_TASK)
language_job_id

## 7. Benchmark languages — fetch results

À lancer quand le batch est terminé. Renomme les colonnes Mistral vers le vocabulaire corpus (`benchmark_languages`, etc.).

In [ ]:
LANGUAGE_JOB_ID = "15f903db-270c-41cc-b202-eee19409ba0f"

lang_df = corpus[["bibkey", "title", "abstract"]].copy()
languages = mistral_batch_results(LANGUAGE_JOB_ID, lang_df, task=LANGUAGE_TASK)

languages = languages.rename(
    columns={
        "mistral_languages": "benchmark_languages",
        "mistral_evidence": "language_evidence",
        "mistral_reasoning": "language_reasoning",
    }
)

out_cols = [
    "bibkey",
    "benchmark_languages",
    "language_evidence",
    "language_reasoning",
]
languages = languages[[c for c in out_cols if c in languages.columns]]
languages.to_parquet(LANGUAGE_OUT_PATH, index=False)
LANGUAGE_OUT_PATH

## 8. Merge extractions into corpus

Met à jour `single_task_benchmark_paper.parquet` en fusionnant resource links et langues benchmark.
La colonne `language` existante (métadonnée ACL, langue du papier) est conservée ; `benchmark_languages` décrit le texte annoté.

In [ ]:
cmd = [
    "uv",
    "run",
    "python",
    "-m",
    "src.corpus.build_single_task_benchmark_paper",
    "--input",
    str(CORPUS_PATH.resolve()),
    "--output",
    str(CORPUS_PATH.resolve()),
    "--resource-links",
    str(LINK_OUT_PATH.resolve()),
    "--language-extractions",
    str(LANGUAGE_OUT_PATH.resolve()),
]
subprocess.run(cmd, check=True, cwd=Path().resolve().parent)

CompletedProcess(args=['uv', 'run', 'python', '-m', 'src.corpus.build_single_task_benchmark_paper', '--input', 'C:\\Users\\samma\\Documents\\Programmation\\UdeM\\nlp-benchmark-taxonomy\\data\\corpus\\single_task_benchmark_paper.parquet', '--output', 'C:\\Users\\samma\\Documents\\Programmation\\UdeM\\nlp-benchmark-taxonomy\\data\\corpus\\single_task_benchmark_paper.parquet', '--resource-links', 'C:\\Users\\samma\\Documents\\Programmation\\UdeM\\nlp-benchmark-taxonomy\\data\\corpus\\resource_links.parquet'], returncode=0)